# BipedalWalker — Backflip Curriculum (v2)

Trains a BipedalWalker agent to perform clean backflips and land on its feet.

## Three-Stage Curriculum

| Stage | Goal | Gravity | Fall penalty | Key rewards |
|-------|------|---------|--------------|-------------|
| 1 | Learn to flip | −5.0 (easy) | Cancelled | Angular speed, rotation progress, milestones |
| 2 | Learn to land upright | −7.5 | Cancelled pre-flip only | Uprightness gradient, knee-crash penalty, clean-landing bonus |
| 3 | Consolidate under real gravity | −10.0 (real) | Not cancelled | Same as Stage 2, higher stakes |

## Key Fixes vs the Original Wrapper
- **Gravity curriculum** — gravity increases each stage toward real physics
- **Fixed landing check** — landing bonus only fires when hull angle < 0.4 rad (upright)
- **Knee-crash penalty** — landing with hull > 0.8 rad gives −100 and terminates the episode
- **Uprightness gradient** — `cos(hull_angle) × 25` guides agent back to vertical post-flip
- **Spin dampening** — angular velocity is penalised after flip completes (stop spinning, land!)
- **In-air tuck reward** — bent knees while descending help prepare for feet-first landing
- **Rotation reward stops after flip** — no incentive to keep spinning after 360°

## How to Run
Run cells top to bottom. Each stage saves a `.zip` model and the next stage loads it.
You can restart from any stage if a save already exists — the training cell will skip.

To monitor training live:
```
tensorboard --logdir ./tb_logs_flipper
```

## 1. Imports & Setup

In [8]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import gymnasium as gym
import numpy as np
import torch

import custom_bipedal  # your local copy of the BipedalWalker environment

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback, CallbackList

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
NUM_ENVS = 32
print(f"Device: {DEVICE}  |  Parallel envs: {NUM_ENVS}")

Device: cuda  |  Parallel envs: 32


## 2. Improved `CurriculumFlipperWrapper`

Run this cell once — every training/test cell below depends on it.

In [9]:
class CurriculumFlipperWrapper(gym.Wrapper):
    """
    5-stage curriculum wrapper for BipedalWalker backflip training.

    Stage 1 — Rotation Mastery  (gravity = -5.0)
        Learn to discover and reliably complete a full backflip.
        Fall penalty cancelled so the agent takes risks.

    Stage 2 — Landing  (gravity = -7.5)
        Land upright on feet after the flip.
        Knee-crash landings penalised and terminated.

    Stage 3 — Consolidation  (gravity = -10.0)
        Same as Stage 2 under full gravity, no hand-holding.

    Stage 4 — Precision Landing  (gravity = -10.0, strict)
        Fixes the Stage-3 problem where the flip completes too close
        to the ground so the legs end up pointing forward.
        Enforces a tighter landing angle (0.22 rad) and stability window.

    Stage 5 — Landing Stabilization  (gravity = -10.0, recovery + straight legs)
        Same as Stage 4 but removes premature knee-crash terminations to allow
        recovery training. Also requires the legs to be straight (extended) at the
        end of the stability window before the landing is successful.
    """

    GRAVITY                = {1: -5.0, 2: -7.5, 3: -10.0, 4: -10.0, 5: -10.0}
    CLEAN_LANDING_ANGLE    = 0.4    # ~23°  stages 1-3
    CLEAN_LANDING_ANGLE_S4 = 0.22   # ~12.6° stage 4
    CLEAN_LANDING_ANGLE_S5 = 0.28   # ~16°  stage 5
    CRASH_LANDING_ANGLE    = 0.8    # ~46°  knee crash (stages 1-4)
    CRASH_LANDING_ANGLE_S5 = 1.1    # ~63°  knee crash (stage 5)
    STABILITY_STEPS        = 45     # steps to hold landing pose (stages 4-5)

    def __init__(self, env, stage: int = 1, max_steps: int = 1500):
        super().__init__(env)
        self.stage     = stage
        self.max_steps = max_steps
        self.cumulative_angle = 0.0
        self.prev_angle       = 0.0
        self.flip_completed   = False
        self.landed           = False
        self.step_counter     = 0
        self._milestone_flags = {}
        self.stable_steps     = 0
        self.max_stable_steps = 0
        low  = np.append(self.env.observation_space.low,  -np.inf)
        high = np.append(self.env.observation_space.high,  np.inf)
        self.observation_space = gym.spaces.Box(low, high, dtype=np.float32)

    # ------------------------------------------------------------------
    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        gravity = self.GRAVITY.get(self.stage, -5.0)
        try:
            self.env.unwrapped.world.gravity = (0.0, float(gravity))
        except Exception:
            pass
        self.cumulative_angle = 0.0
        self.prev_angle       = obs[0]
        self.flip_completed   = False
        self.landed           = False
        self.step_counter     = 0
        self._milestone_flags = {}
        self.stable_steps     = 0
        self.max_stable_steps = 0
        return np.append(obs, 0.0).astype(np.float32), info

    # ------------------------------------------------------------------
    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)
        self.step_counter += 1

        # ── Angle tracking ──────────────────────────────────────────────
        current_angle = obs[0]
        delta_angle   = current_angle - self.prev_angle
        if delta_angle >  np.pi: delta_angle -= 2 * np.pi
        if delta_angle < -np.pi: delta_angle += 2 * np.pi
        prev_cumulative        = self.cumulative_angle
        self.cumulative_angle += delta_angle
        self.prev_angle        = current_angle
        abs_angle = abs(self.cumulative_angle)
        abs_prev  = abs(prev_cumulative)

        # ── Observation shorthands ──────────────────────────────────────
        hull_angle  = obs[0]            # 0=upright, ±π=upside-down
        ang_vel     = obs[1]            # hull angular velocity
        vel_x       = obs[2]            # normalised horizontal velocity
        vel_y       = obs[3]            # normalised vertical velocity (pos=up)
        foot1_down  = obs[8]  == 1.0
        foot2_down  = obs[13] == 1.0
        feet_contact = foot1_down or foot2_down
        both_feet    = foot1_down and foot2_down
        in_air       = not foot1_down and not foot2_down
        is_falling   = (base_reward == -100)
        knee1_angle  = obs[6]           # 0=extended, ~2=tucked
        knee2_angle  = obs[11]
        hip1_angle   = obs[4]           # leg 1 hip angle
        hip2_angle   = obs[9]           # leg 2 hip angle
        height_frac  = obs[14]          # ray-0 lidar: 0=touching ground, 1=~5.3m up

        custom_reward = 0.0

        # ══════════════════════════════════════════════════════════════
        # PRE-FLIP rewards
        # ══════════════════════════════════════════════════════════════
        if not self.flip_completed:
            custom_reward += abs(ang_vel) * 5.0               # spin faster
            custom_reward += (abs_angle - abs_prev) * 15.0   # rotation progress
            if in_air:
                custom_reward += 1.0                          # airtime bonus
            # HEIGHT FIX 1: reward upward velocity (stages 3+)
            # Encourages jumping BEFORE spinning to gain altitude first.
            if self.stage >= 3 and vel_y > 0:
                custom_reward += vel_y * 10.0
            for ms in [np.pi / 2, np.pi, 3 * np.pi / 2]:
                key = f"ms_{ms:.4f}"
                if not self._milestone_flags.get(key, False) and abs_angle >= ms:
                    self._milestone_flags[key] = True
                    custom_reward += 50.0

        # ══════════════════════════════════════════════════════════════
        # FLIP COMPLETION (full 2π rotation)
        # ══════════════════════════════════════════════════════════════
        if abs_angle >= 2 * np.pi and not self.flip_completed:
            self.flip_completed = True
            custom_reward += 300.0
            # HEIGHT FIX 2: large altitude bonus at flip completion (stages 3+)
            # obs[14]=straight-down lidar fraction: 0=ground, 1=~5.3m above.
            # Higher flip completion → more room for legs to extend downward.
            if self.stage >= 3:
                custom_reward += height_frac * 200.0

        # ══════════════════════════════════════════════════════════════
        # POST-FLIP rewards (flip done, waiting to land)
        # ══════════════════════════════════════════════════════════════
        if self.flip_completed and not self.landed:

            # 1. Uprightness gradient
            custom_reward += np.cos(hull_angle) * 25.0

            # 2. Dampen residual spin
            custom_reward -= abs(ang_vel) * 5.0

            if in_air:
                # 3. In-air tuck (stages 1-2 only)
                if self.stage < 3:
                    custom_reward += (knee1_angle + knee2_angle) * 1.5

                # HEIGHT FIX 3: per-step altitude reward while airborne (stages 3+)
                # Keeps the agent high so it has TIME to orient legs before impact.
                if self.stage >= 3:
                    custom_reward += height_frac * 4.0

                # Stage 4/5: in-air landing prep
                if self.stage in [4, 5]:
                    if self.stable_steps > 0:
                        custom_reward -= 30.0       # penalty for breaking stability
                        self.stable_steps = 0       # reset if went airborne mid-window
                    # a) Hip-down: legs hanging toward ground
                    hip_posture = 2.0 - abs(obs[4]) - abs(obs[9])
                    custom_reward += hip_posture * 3.0
                    # b) Leg extension: straight legs absorb impact better
                    custom_reward += (4.0 - knee1_angle - knee2_angle) * 2.0

            # ── Landing angle threshold ──────────────────────────────
            clean_angle = self.CLEAN_LANDING_ANGLE_S5 if self.stage == 5 \
                          else (self.CLEAN_LANDING_ANGLE_S4 if self.stage == 4 else self.CLEAN_LANDING_ANGLE)
            crash_angle = self.CRASH_LANDING_ANGLE_S5 if self.stage == 5 else self.CRASH_LANDING_ANGLE

            # 4. KNEE-CRASH CHECK
            if feet_contact and abs(hull_angle) > crash_angle:
                if self.stage <= 4:
                    penalty = -100.0 if self.stage <= 2 else (-150.0 if self.stage == 3 else -200.0)
                    custom_reward    += penalty
                    self.stable_steps = 0
                    terminated        = True
                else: # Stage 5: do NOT terminate! Let it try to recover.
                    custom_reward    -= 10.0  # penalty for bad posture
                    self.stable_steps = 0

            # 5. CLEAN LANDING / STABILITY WINDOW
            elif feet_contact and abs(hull_angle) < clean_angle:
                if self.stage <= 2:
                    custom_reward += 1000.0 * (1.0 - (abs(hull_angle) / clean_angle))
                    self.landed = True;  terminated = True

                elif self.stage == 3:
                    custom_reward += 2000.0 * (1.0 - (abs(hull_angle) / clean_angle))
                    self.landed = True;  terminated = True

                else: # self.stage in [4, 5]
                    # Increment stability counter while at least 1 foot has contact and hull is upright
                    self.stable_steps += 1
                    uprightness = 1.0 - (abs(hull_angle) / clean_angle)
                    custom_reward += uprightness * 20.0   # strong reward for standing upright
                    custom_reward -= abs(ang_vel) * 10.0  # no spinning
                    custom_reward -= abs(vel_x)   * 8.0   # no sliding
                    custom_reward -= abs(vel_y)   * 8.0   # no bouncing / hopping

                    # Stage 5: reward straight legs and leg spread (triangle/split stance)
                    if self.stage == 5:
                        legs_straightness = 4.0 - (knee1_angle + knee2_angle)
                        custom_reward += legs_straightness * 5.0  # strong incentive to extend legs
                        
                        # Encourage split stance (legs spread out like a triangle to increase stability)
                        hip_spread = abs(hip1_angle - hip2_angle)
                        custom_reward += min(hip_spread, 0.6) * 10.0  # reward up to +6.0 for spreading legs

                    # Mid-window milestone at half-way to give a shaping signal
                    half_window = self.STABILITY_STEPS // 2
                    if self.stable_steps == half_window:
                        custom_reward += 500.0 * uprightness  # big intermediate reward

                    # COMPLETION: stable for full window with feet on the ground
                    # Stage 4: requires both feet; Stage 5: only 1 foot needed (more forgiving)
                    landing_feet = both_feet if self.stage == 4 else feet_contact
                    if self.stable_steps >= self.STABILITY_STEPS and landing_feet:
                        # Straight-leg bonus for Stage 5 (reward-only, not gating)
                        knee_bonus = 0.0
                        if self.stage == 5:
                            avg_knee = (knee1_angle + knee2_angle) / 2.0
                            knee_bonus = max(0.0, (0.7 - avg_knee)) * 500.0  # up to +350 for straight legs
                        custom_reward -= max(0.0, -vel_y) * 50.0  # impact penalty
                        custom_reward += 3000.0 * uprightness + knee_bonus
                        self.landed   = True
                        terminated    = True

            # 6. BREAKING STABILITY / RESET (Stage 4/5 only)
            elif self.stage in [4, 5]:
                # Hull angle not clean and not in crash zone: penalise and reset counter
                if self.stable_steps > 0:
                    custom_reward -= 20.0  # softer penalty (was 30) so partial progress still counts
                    self.stable_steps = 0

        # Update peak stable steps
        if self.stable_steps > self.max_stable_steps:
            self.max_stable_steps = self.stable_steps

        # ══════════════════════════════════════════════════════════════
        # STAGE-SPECIFIC EXTRAS
        # ══════════════════════════════════════════════════════════════
        if self.stage == 1:
            if is_falling:
                custom_reward += 100.0
        elif self.stage == 2:
            if is_falling and not self.flip_completed:
                custom_reward += 50.0
        elif self.stage >= 3:
            pass  # real gravity, no hand-holding

        if self.step_counter >= self.max_steps:
            truncated = True

        info["flip_completed"]   = self.flip_completed
        info["landed"]           = self.landed
        info["cumulative_angle"] = self.cumulative_angle
        info["abs_angle_deg"]    = np.degrees(abs_angle)
        info["max_stable_steps"] = self.max_stable_steps
        info["knee1_angle"]      = knee1_angle
        info["knee2_angle"]      = knee2_angle

        obs_out = np.append(obs, abs_angle / (2 * np.pi)).astype(np.float32)
        return obs_out, base_reward + custom_reward, terminated, truncated, info


print("CurriculumFlipperWrapper defined ✓")

CurriculumFlipperWrapper defined ✓


## 3. Callbacks

In [10]:
class FlipMetricsCallback(BaseCallback):
    """Logs flip/landing success rates and Stage 5 metrics to TensorBoard per episode."""
    def __init__(self, window=200, verbose=0):
        super().__init__(verbose)
        self.window = window
        self._flip:        list = []
        self._landed:      list = []
        self._rot_deg:     list = []
        self._stable:      list = []
        self._knee_bend:   list = []

    def _on_step(self) -> bool:
        dones = self.locals.get("dones")
        for i, info in enumerate(self.locals.get("infos", [])):
            if dones is not None and dones[i]:
                self._flip.append(float(info.get("flip_completed", False)))
                self._landed.append(float(info.get("landed", False)))
                self._rot_deg.append(info.get("abs_angle_deg", 0.0))
                self._stable.append(float(info.get("max_stable_steps", 0)))
                
                k1 = info.get("knee1_angle", 0.0)
                k2 = info.get("knee2_angle", 0.0)
                self._knee_bend.append(float(k1 + k2) / 2.0)

                for buf in (self._flip, self._landed, self._rot_deg, self._stable, self._knee_bend):
                    if len(buf) > self.window:
                        buf.pop(0)

        if self._flip:
            self.logger.record("flip/success_rate",        np.mean(self._flip))
            self.logger.record("flip/landing_rate",        np.mean(self._landed))
            self.logger.record("flip/avg_rotation_deg",    np.mean(self._rot_deg))
            self.logger.record("flip/avg_max_stable_steps", np.mean(self._stable))
            self.logger.record("flip/avg_knee_bend",       np.mean(self._knee_bend))
        return True


class RenderCallback(BaseCallback):
    """Periodically renders the current policy for visual inspection."""
    def __init__(self, render_freq=100_000, stage=1, verbose=0):
        super().__init__(verbose)
        self.render_freq = render_freq
        self.stage = stage
        self._env = None

    def _on_step(self) -> bool:
        if self.num_timesteps > 0 and self.num_timesteps % self.render_freq == 0:
            print(f"\n[Step {self.num_timesteps:,}] Rendering policy...")
            if self._env is None:
                e = custom_bipedal.BipedalWalker(render_mode="human")
                self._env = CurriculumFlipperWrapper(e, stage=self.stage)
            obs, _ = self._env.reset()
            done = False
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, _, terminated, truncated, _ = self._env.step(action)
                done = terminated or truncated
            print("Render done. Resuming training.\n")
        return True


print("Callbacks defined ✓")

Callbacks defined ✓


## 4. Stage 1 — Rotation Mastery

**Gravity:** −5.0 (easy to get airborne and spin)  
**Goal:** reliably complete a full 360° backflip  
**Duration:** ~5 million steps  

If `ppo_flipper_stage1.zip` already exists, this cell is skipped automatically.

In [16]:
STAGE1_SAVE  = "ppo_flipper_stage1"
STAGE1_STEPS = 5_000_000

if os.path.exists(f"{STAGE1_SAVE}.zip"):
    print(f"Stage 1 model already exists ({STAGE1_SAVE}.zip) — skipping training.")
    print("Delete the file and re-run this cell to retrain from scratch.")
else:
    print("Stage 1: Rotation Mastery  (gravity = -5.0)")
    print("To monitor: tensorboard --logdir ./tb_logs_flipper\n")

    def make_env_s1():
        def _init():
            e = custom_bipedal.BipedalWalker(hardcore=False)
            return CurriculumFlipperWrapper(e, stage=1)
        return _init

    vec_env_s1 = SubprocVecEnv([make_env_s1() for _ in range(NUM_ENVS)])

    model_s1 = PPO(
        "MlpPolicy", vec_env_s1,
        verbose=1,
        device=DEVICE,
        n_steps=2048,
        batch_size=8192,
        n_epochs=5,
        learning_rate=3e-4,
        gae_lambda=0.95,
        gamma=0.99,
        clip_range=0.2,
        ent_coef=0.05,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
        tensorboard_log="./tb_logs_flipper",
    )

    cbs_s1 = CallbackList([
        FlipMetricsCallback(window=200)
    ])

    model_s1.learn(total_timesteps=STAGE1_STEPS, callback=cbs_s1, progress_bar=True)
    model_s1.save(STAGE1_SAVE)
    vec_env_s1.close()
    print(f"\nStage 1 complete. Saved to {STAGE1_SAVE}.zip")

Stage 1 model already exists (ppo_flipper_stage1.zip) — skipping training.
Delete the file and re-run this cell to retrain from scratch.


## 5. Stage 2 — Landing

**Gravity:** −7.5 (harder — landing sloppily actually hurts)  
**Goal:** land upright on feet after the flip  
**Duration:** ~4 million steps  

Loads Stage 1 weights and continues training with a lower learning rate.  
**New in this stage:** knee-crash penalty, clean-landing bonus, uprightness gradient.

In [ ]:
STAGE2_SAVE  = "ppo_flipper_stage2"
STAGE2_STEPS = 5_000_000

if os.path.exists(f"{STAGE2_SAVE}.zip"):
    print(f"Stage 2 model already exists ({STAGE2_SAVE}.zip) — skipping training.")
else:
    if not os.path.exists(f"{STAGE1_SAVE}.zip"):
        raise FileNotFoundError(f"Stage 1 model not found: {STAGE1_SAVE}.zip\nRun Stage 1 first.")

    print("Stage 2: Landing  (gravity = -7.5)")
    print("To monitor: tensorboard --logdir ./tb_logs_flipper\n")

    def make_env_s2():
        def _init():
            e = custom_bipedal.BipedalWalker(hardcore=False)
            return CurriculumFlipperWrapper(e, stage=2)
        return _init

    vec_env_s2 = SubprocVecEnv([make_env_s2() for _ in range(NUM_ENVS)])

    # Load Stage 1 weights, plug into Stage 2 env
    model_s2 = PPO.load(
        STAGE1_SAVE, env=vec_env_s2, device=DEVICE,
        tensorboard_log="./tb_logs_flipper"
    )
    # Fine-tune with lower LR and less entropy
    model_s2.learning_rate = 1e-4
    model_s2.ent_coef      = 0.005

    cbs_s2 = CallbackList([
        FlipMetricsCallback(window=200)
    ])

    model_s2.learn(
        total_timesteps=STAGE2_STEPS, callback=cbs_s2,
        progress_bar=True, reset_num_timesteps=False
    )
    model_s2.save(STAGE2_SAVE)
    vec_env_s2.close()
    print(f"\nStage 2 complete. Saved to {STAGE2_SAVE}.zip")

## 6. Stage 3 — Consolidation (Real Gravity)

**Gravity:** −10.0 (real Earth gravity)  
**Goal:** perform and land clean backflips under realistic physics  
**Duration:** ~3 million steps  

No fall penalty assistance. Higher stakes for both success and failure.

In [ ]:
STAGE2_SAVE = "ppo_flipper_stage2"
STAGE3_SAVE  = "ppo_flipper_stage3"
STAGE3_STEPS = 5_000_000

if os.path.exists(f"{STAGE3_SAVE}.zip"):
    print(f"Stage 3 model already exists ({STAGE3_SAVE}.zip) — skipping training.")
else:
    if not os.path.exists(f"{STAGE2_SAVE}.zip"):
        raise FileNotFoundError(f"Stage 2 model not found: {STAGE2_SAVE}.zip\nRun Stage 2 first.")

    print("Stage 3: Real-gravity consolidation  (gravity = -10.0)")
    print("To monitor: tensorboard --logdir ./tb_logs_flipper\n")

    def make_env_s3():
        def _init():
            e = custom_bipedal.BipedalWalker(hardcore=False)
            return CurriculumFlipperWrapper(e, stage=3)
        return _init

    vec_env_s3 = SubprocVecEnv([make_env_s3() for _ in range(NUM_ENVS)])

    model_s3 = PPO.load(
        STAGE2_SAVE, env=vec_env_s3, device=DEVICE,
        tensorboard_log="./tb_logs_flipper"
    )
    model_s3.learning_rate = 5e-5
    model_s3.ent_coef      = 0.001

    cbs_s3 = CallbackList([
        FlipMetricsCallback(window=200)
    ])

    model_s3.learn(
        total_timesteps=STAGE3_STEPS, callback=cbs_s3,
        progress_bar=True, reset_num_timesteps=False
    )
    model_s3.save(STAGE3_SAVE)
    vec_env_s3.close()
    print(f"\nStage 3 complete. Saved to {STAGE3_SAVE}.zip")

## 7. Stage 4 — Precision Landing

Fine-tunes landing quality from the Stage 3 model (gravity stays -10.0).

**Root-cause fix**: after Stage 3 the flip completes too low so legs point forward.  
Height rewards (active from Stage 3 in the wrapper) incentivise a higher trajectory.

**Stage 4 landing requirements vs Stage 3:**
- Both feet must touch **simultaneously**
- Hull angle < **0.22 rad (~12.6°)**
- **Stability window (30 steps)**: must hold the pose before the bonus fires  — rewards *standing*, not just touching the ground
- Per-step `+uprightness×15` while holding the pose
- Velocity damping: no spinning or sliding through the window
- Bonus **+3000**, crash penalty **−200**

In [ ]:
STAGE3_SAVE  = "ppo_flipper_stage3"
STAGE4_SAVE  = "ppo_flipper_stage4"
STAGE4_STEPS = 15_000_000

if os.path.exists(f"{STAGE4_SAVE}.zip"):
    print(f"Stage 4 model already exists ({STAGE4_SAVE}.zip) — skipping training.")
else:
    if not os.path.exists(f"{STAGE3_SAVE}.zip"):
        raise FileNotFoundError(f"Stage 3 model not found: {STAGE3_SAVE}.zip\nRun Stage 3 first.")

    print("Stage 4: Precision landing with height rewards  (gravity = -10.0)")
    print("To monitor: tensorboard --logdir ./tb_logs_flipper\n")

    def make_env_s4():
        def _init():
            e = custom_bipedal.BipedalWalker(hardcore=False)
            return CurriculumFlipperWrapper(e, stage=4)
        return _init

    vec_env_s4 = SubprocVecEnv([make_env_s4() for _ in range(NUM_ENVS)])

    model_s4 = PPO.load(
        STAGE3_SAVE, env=vec_env_s4, device=DEVICE,
        tensorboard_log="./tb_logs_flipper"
    )
    # Higher entropy than Stage 3 — agent needs to explore landing strategies
    # and escape reward-hacking local optima.
    model_s4.learning_rate = 5e-5
    model_s4.ent_coef      = 0.002

    cbs_s4 = CallbackList([
        FlipMetricsCallback(window=200)
    ])

    model_s4.learn(
        total_timesteps=STAGE4_STEPS, callback=cbs_s4,
        progress_bar=True, reset_num_timesteps=False
    )
    model_s4.save(STAGE4_SAVE)
    vec_env_s4.close()
    print(f"\nStage 4 complete. Saved to {STAGE4_SAVE}.zip")

## 8. Stage 5 — Landing Stabilization

In this stage, we load from the Stage 4 checkpoint (`ppo_flipper_stage4.zip`) and train with a relaxed landing-crash angle of `1.1` rad (~63°) and a slightly relaxed clean landing angle of `0.28` rad (~16°).
This allows the agent to land on one foot while still tilted, absorb the landing impact, and use its joints to pull itself upright into the 30-step standing stabilization window.

In [16]:
STAGE4_SAVE  = "ppo_flipper_stage4"
STAGE5_SAVE  = "ppo_flipper_stage5"
STAGE5_STEPS = 3_000_000

if os.path.exists(f"{STAGE5_SAVE}.zip"):
    print(f"Stage 5 model already exists ({STAGE5_SAVE}.zip) — skipping training.")
else:
    if not os.path.exists(f"{STAGE4_SAVE}.zip"):
        raise FileNotFoundError(f"Stage 4 model not found: {STAGE4_SAVE}.zip\nRun Stage 4 first.")

    print("Stage 5: Landing stabilization with relaxed crash angle (gravity = -10.0)")
    print("To monitor: tensorboard --logdir ./tb_logs_flipper\n")

    def make_env_s5():
        def _init():
            e = custom_bipedal.BipedalWalker(hardcore=False)
            return CurriculumFlipperWrapper(e, stage=5)
        return _init

    vec_env_s5 = SubprocVecEnv([make_env_s5() for _ in range(NUM_ENVS)])

    model_s5 = PPO.load(
        STAGE4_SAVE, env=vec_env_s5, device=DEVICE,
        tensorboard_log="./tb_logs_flipper"
    )
    model_s5.learning_rate = 3e-4
    model_s5.ent_coef      = 0.05

    cbs_s5 = CallbackList([
        FlipMetricsCallback(window=200)
    ])

    model_s5.learn(
        total_timesteps=STAGE5_STEPS, callback=cbs_s5,
        progress_bar=True, reset_num_timesteps=False
    )
    model_s5.save(STAGE5_SAVE)
    vec_env_s5.close()
    print(f"\nStage 5 complete. Saved to {STAGE5_SAVE}.zip")

Stage 5: Landing stabilization with relaxed crash angle (gravity = -10.0)
To monitor: tensorboard --logdir ./tb_logs_flipper

Logging to ./tb_logs_flipper\PPO_3


Output()

c:\Users\franc\Documents\FEUP\MEST\1ANO\2SEM\ASMA\ASMA2\venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------------
| flip/                   |           |
|    avg_knee_bend        | 0.428     |
|    avg_max_stable_steps | 0         |
|    avg_rotation_deg     | 388.19604 |
|    landing_rate         | 0         |
|    success_rate         | 0.92      |
| time/                   |           |
|    fps                  | 3995      |
|    iterations           | 1         |
|    time_elapsed         | 16        |
|    total_timesteps      | 30212096  |
---------------------------------------


-------------------------------------------
| flip/                   |               |
|    avg_knee_bend        | 0.392         |
|    avg_max_stable_steps | 0             |
|    avg_rotation_deg     | 376.95172     |
|    landing_rate         | 0             |
|    success_rate         | 0.93          |
| time/                   |               |
|    fps                  | 4050          |
|    iterations           | 2             |
|    time_elapsed         | 32            |
|    total_timesteps      | 30277632      |
| train/                  |               |
|    approx_kl            | 5.0107818e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -8.98         |
|    explained_variance   | 0.412         |
|    learning_rate        | 5e-05         |
|    loss                 | 1.64e+04      |
|    n_updates            | 2305          |
|    policy_gradient_loss | -5.34e-05     |
|    std                  | 2.29

...

---------------------------------------
| flip/                   |           |
|    avg_knee_bend        | 0.405     |
|    avg_max_stable_steps | 0         |
|    avg_rotation_deg     | 381.48642 |
|    landing_rate         | 0         |
|    success_rate         | 0.865     |
| time/                   |           |
|    fps                  | 4224      |
|    iterations           | 43        |
|    time_elapsed         | 667       |
|    total_timesteps      | 32964608  |
| train/                  |           |
|    approx_kl            | 0.000166  |
|    clip_fraction        | 0         |
|    clip_range           | 0.2       |
|    entropy_loss         | -9.11     |
|    explained_variance   | 0.606     |
|    learning_rate        | 5e-05     |
|    loss                 | 1.88e+04  |
|    n_updates            | 2510      |
|    policy_gradient_loss | -5.92e-05 |
|    std                  | 2.37      |
|    value_loss           | 3.83e+04  |
---------------------------------------


------------------------------------------
| flip/                   |              |
|    avg_knee_bend        | 0.396        |
|    avg_max_stable_steps | 0            |
|    avg_rotation_deg     | 376.91202    |
|    landing_rate         | 0            |
|    success_rate         | 0.91         |
| time/                   |              |
|    fps                  | 4231         |
|    iterations           | 44           |
|    time_elapsed         | 681          |
|    total_timesteps      | 33030144     |
| train/                  |              |
|    approx_kl            | 2.653422e-05 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -9.12        |
|    explained_variance   | 0.614        |
|    learning_rate        | 5e-05        |
|    loss                 | 1.89e+04     |
|    n_updates            | 2515         |
|    policy_gradient_loss | -1.16e-05    |
|    std                  | 2.37         |
|    value_

-------------------------------------------
| flip/                   |               |
|    avg_knee_bend        | 0.39          |
|    avg_max_stable_steps | 0             |
|    avg_rotation_deg     | 379.19156     |
|    landing_rate         | 0             |
|    success_rate         | 0.865         |
| time/                   |               |
|    fps                  | 4238          |
|    iterations           | 45            |
|    time_elapsed         | 695           |
|    total_timesteps      | 33095680      |
| train/                  |               |
|    approx_kl            | 2.1898028e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -9.12         |
|    explained_variance   | 0.617         |
|    learning_rate        | 5e-05         |
|    loss                 | 1.95e+04      |
|    n_updates            | 2520          |
|    policy_gradient_loss | -2.55e-05     |
|    std                  | 2.37

------------------------------------------
| flip/                   |              |
|    avg_knee_bend        | 0.381        |
|    avg_max_stable_steps | 0            |
|    avg_rotation_deg     | 374.7924     |
|    landing_rate         | 0            |
|    success_rate         | 0.86         |
| time/                   |              |
|    fps                  | 4243         |
|    iterations           | 46           |
|    time_elapsed         | 710          |
|    total_timesteps      | 33161216     |
| train/                  |              |
|    approx_kl            | 0.0001452911 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -9.12        |
|    explained_variance   | 0.632        |
|    learning_rate        | 5e-05        |
|    loss                 | 1.82e+04     |
|    n_updates            | 2525         |
|    policy_gradient_loss | -5.5e-05     |
|    std                  | 2.38         |
|    value_


Stage 5 complete. Saved to ppo_flipper_stage5.zip


## 9. Test the Final Model

Loads the best available saved model and runs 5 rendered test episodes.  
Priority: Stage 5 > Stage 4 > Stage 3 > Stage 2 > Stage 1

In [23]:
import time

# Pick the best available model  (Stage 5 > 4 > 3 > 2 > 1)
for path, stage in [
    ("ppo_flipper_stage5", 5),
    ("ppo_flipper_stage4", 4),
    ("ppo_flipper_stage3", 3),
    ("ppo_flipper_stage2", 2),
    ("ppo_flipper_stage1", 1),
]:
    if os.path.exists(f"{path}.zip"):
        model_path, model_stage = path, stage
        break
else:
    raise FileNotFoundError("No saved model found. Train at least Stage 1 first.")

print(f"Loading model: {model_path}.zip  (Stage {model_stage})")
model_test = PPO.load(model_path, device=DEVICE)

env_test = custom_bipedal.BipedalWalker(hardcore=False, render_mode="human")
env_test = CurriculumFlipperWrapper(env_test, stage=model_stage)

NUM_TEST_EPISODES = 5

for ep in range(1, NUM_TEST_EPISODES + 1):
    obs, _ = env_test.reset()
    done = False
    total_reward = 0.0
    steps = 0
    while not done:
        action, _ = model_test.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env_test.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1
    print(f"Episode {ep:2d}  |  Steps: {steps:4d}  |  Reward: {total_reward:8.1f}  |  "
          f"Flip: {chr(10003) if info['flip_completed'] else chr(10007)}  |  "
          f"Landed: {chr(10003) if info['landed'] else chr(10007)}  |  "
          f"Rotation: {info['abs_angle_deg']:.1f}°")
    time.sleep(0.5)

env_test.close()

Loading model: ppo_flipper_stage4.zip  (Stage 4)
Episode  1  |  Steps:   86  |  Reward:    787.7  |  Flip: ✓  |  Landed: ✗  |  Rotation: 389.9°
Episode  2  |  Steps:   86  |  Reward:    787.8  |  Flip: ✓  |  Landed: ✗  |  Rotation: 392.1°
Episode  3  |  Steps:   85  |  Reward:    825.5  |  Flip: ✓  |  Landed: ✗  |  Rotation: 400.2°
Episode  4  |  Steps:   86  |  Reward:    788.8  |  Flip: ✓  |  Landed: ✗  |  Rotation: 390.7°
Episode  5  |  Steps:   86  |  Reward:    788.7  |  Flip: ✓  |  Landed: ✗  |  Rotation: 391.4°
